In [1]:
import pandas as pd
import numpy as np

In [2]:
df_all = pd.read_csv('Cleaned_data.csv')

In [3]:
df_all

,Campaign_ID,Campaign_Type,Target_Audience,Duration,Channel_Used,Impressions,Clicks,Leads,Conversions,Revenue,Acquisition_Cost,ROI,Language,Engagement_Score,Customer_Segment,Date,Total_Acquisition_Cost,Net_Profit,Profit_Loss
0,NY-CMP-1000,Social Media,College Students,21.000000,"WhatsApp, YouTube",57804.0,6156.000000,3616.0,2355.000000,1867515.0,162.31857,3.885455,Hindi,20.980000,College Students,29-04-2025,382260.232841,1.485255e+06,1
1,NY-CMP-1001,Paid Ads,Tier 2 City Customers,18.000000,YouTube,91801.0,3321.000000,1971.0,1357.000000,1046247.0,180.83000,3.263673,Hindi,7.240000,College Students,06-04-2025,245386.310000,8.008607e+05,1
2,NY-CMP-1002,Influencer,Youth,23.000000,"WhatsApp, Google, YouTube",15536.0,2182.000000,952.0,755.000000,197055.0,90.60000,1.880795,English,25.030000,College Students,14-01-2025,68403.000000,1.286520e+05,1
3,NY-CMP-1004,Paid Ads,College Students,10.000000,"Facebook, Instagram",96871.0,3743.000000,2060.0,1258.000000,518296.0,228.60000,0.802275,Hindi,7.290000,Tier 2 City Customers,29-12-2024,287578.800000,2.307172e+05,1
4,NY-CMP-1006,Paid Ads,College Students,21.000000,"YouTube, Google, WhatsApp",82267.0,5458.378102,1873.0,1386.000000,633402.0,210.17000,1.174430,Tamil,9.906739,Youth,01-09-2024,291295.620000,3.421064e+05,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118950,TI-CMP-56545,Social Media,College Students,7.000000,"Facebook, Instagram, YouTube",78207.0,3086.000000,883.0,620.000000,199020.0,362.11000,-0.113529,English,5.870000,College Students,01-03-2025,224508.200000,-2.548820e+04,0
118951,TI-CMP-56546,Social Media,Premium Shoppers,14.000000,Instagram,38933.0,2683.000000,1307.0,931.000000,641459.0,298.63000,1.307203,Hindi,13.857298,Premium Shoppers,25-09-2024,278024.530000,3.634345e+05,1
118952,TI-CMP-56548,Influencer,Premium Shoppers,15.000000,"WhatsApp, YouTube, Google",25929.0,1388.000000,808.0,567.000000,408240.0,258.43000,1.786054,Hindi,10.660000,Working Women,19-05-2025,146529.810000,2.617102e+05,1
118953,TI-CMP-56551,SEO,Working Women,20.000000,"Google, Instagram",54886.0,1578.000000,634.0,296.082674,136514.0,436.24000,0.056912,English,4.660000,Working Women,25-08-2024,129163.105647,7.350894e+03,1


In [4]:
# 1. Multi-Label Encoding for Channel_Used (e.g., 'WhatsApp, YouTube' -> separate columns)
channel_dummies = df_all['Channel_Used'].str.get_dummies(sep=', ')
df_all = pd.concat([df_all, channel_dummies], axis=1)

In [11]:
# 2. Convert Date and extract numeric seasonal features
df_all['Date'] = pd.to_datetime(df_all['Date'], format='%d-%m-%Y')
df_all['Campaign_Month'] = df_all['Date'].dt.month
df_all['Campaign_DayOfWeek'] = df_all['Date'].dt.dayofweek

In [6]:
# 3. One-Hot Encoding for standard categorical columns
categorical_cols = ['Campaign_Type', 'Target_Audience', 'Language', 'Customer_Segment']
df_encoded = pd.get_dummies(df_all, columns=categorical_cols, drop_first=True, dtype=int)

In [7]:
# 4. Clean up and drop columns we no longer need for math modeling
columns_to_drop = ['Campaign_ID', 'Date', 'Channel_Used']
df_final = df_encoded.drop(columns=columns_to_drop, errors='ignore')


In [8]:
# 5. Verify the results
print(f"Dataset upgraded! Final structure: {df_final.shape[0]} rows and {df_final.shape[1]} numeric features.")
df_final.head()

Dataset upgraded! Final structure: 118955 rows and 35 numeric features.


,Duration,Impressions,Clicks,Leads,Conversions,Revenue,Acquisition_Cost,ROI,Engagement_Score,Total_Acquisition_Cost,...,Target_Audience_Tier 2 City Customers,Target_Audience_Working Women,Target_Audience_Youth,Language_English,Language_Hindi,Language_Tamil,Customer_Segment_Premium Shoppers,Customer_Segment_Tier 2 City Customers,Customer_Segment_Working Women,Customer_Segment_Youth
0,21.0,57804.0,6156.000000,3616.0,2355.0,1867515.0,162.31857,3.885455,20.980000,382260.232841,...,0,0,0,0,1,0,0,0,0,0
1,18.0,91801.0,3321.000000,1971.0,1357.0,1046247.0,180.83000,3.263673,7.240000,245386.310000,...,1,0,0,0,1,0,0,0,0,0
2,23.0,15536.0,2182.000000,952.0,755.0,197055.0,90.60000,1.880795,25.030000,68403.000000,...,0,0,1,1,0,0,0,0,0,0
3,10.0,96871.0,3743.000000,2060.0,1258.0,518296.0,228.60000,0.802275,7.290000,287578.800000,...,0,0,0,0,1,0,0,1,0,0
4,21.0,82267.0,5458.378102,1873.0,1386.0,633402.0,210.17000,1.174430,9.906739,291295.620000,...,0,0,0,0,0,1,0,0,0,1


In [10]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118955 entries, 0 to 118954
Data columns (total 35 columns):
 #   Column                                  Non-Null Count   Dtype  
---  ------                                  --------------   -----  
 0   Duration                                118955 non-null  float64
 1   Impressions                             118955 non-null  float64
 2   Clicks                                  118955 non-null  float64
 3   Leads                                   118955 non-null  float64
 4   Conversions                             118955 non-null  float64
 5   Revenue                                 118955 non-null  float64
 6   Acquisition_Cost                        118955 non-null  float64
 7   ROI                                     118955 non-null  float64
 8   Engagement_Score                        118955 non-null  float64
 9   Total_Acquisition_Cost                  118955 non-null  float64
 10  Net_Profit                              1189

In [12]:
# 1. Click-Through Rate (CTR): Measures the percentage of people who clicked the ad after seeing it.
# High CTR indicates highly engaging ad creatives or excellent targeting.
df_final['CTR'] = df_final['Clicks'] / (df_final['Impressions'] + 1e-5)

In [13]:
# 2. Conversion Rate (CR): Measures the percentage of clicks that resulted in a successful conversion/purchase.
# High CR implies a strong landing page and highly motivated buyers.
df_final['Conversion_Rate'] = df_final['Conversions'] / (df_final['Clicks'] + 1e-5)

In [14]:
# 3. Cost Per Click (CPC): The average amount of money spent for every individual click received.
# Lower CPC indicates cost-efficient traffic generation.
df_final['CPC'] = df_final['Total_Acquisition_Cost'] / (df_final['Clicks'] + 1e-5)

In [15]:
# 4. Cost Per Lead (CPL): The average budget spent to acquire a single interested prospect (lead).
# Helps assess the financial efficiency of the top-of-funnel marketing activities.
df_final['CPL'] = df_final['Total_Acquisition_Cost'] / (df_final['Leads'] + 1e-5)

In [16]:
# Verify the shape and check the new ratio columns
print(f"Features engineered successfully! New shape: {df_final.shape}")
df_final[['Impressions', 'Clicks', 'CTR', 'Conversions', 'Conversion_Rate', 'CPC', 'CPL']].head()

Features engineered successfully! New shape: (118955, 39)


,Impressions,Clicks,CTR,Conversions,Conversion_Rate,CPC,CPL
0,57804.0,6156.000000,0.106498,2355.0,0.382554,62.095554,105.713560
1,91801.0,3321.000000,0.036176,1357.0,0.408612,73.889283,124.498381
2,15536.0,2182.000000,0.140448,755.0,0.346013,31.348762,71.851890
3,96871.0,3743.000000,0.038639,1258.0,0.336094,76.831098,139.601359
4,82267.0,5458.378102,0.066350,1386.0,0.253922,53.366699,155.523555


In [1]:
df_final.to_csv('Data_for_modelling.csv', index='False')

NameError: name 'df_final' is not defined